In [ ]:
!uv add  datasets
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer
from transformers import AutoModelForSeq2SeqLM, TrainingArguments
from transformers import Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq,Seq2SeqTrainer



In [ ]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
print(dataset["train"][0])


In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")
def tokenize_function(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=150,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names

)


In [ ]:
!uv add "transformers[torch]"
!uv add "accelerate>=1.1.0"
!uv pip show accelerate
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

training_args = Seq2SeqTrainingArguments(
    output_dir="./",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=1,

    fp16=True, 
    dataloader_num_workers=2,

    logging_steps=100,

    save_total_limit=2,

    load_best_model_at_end=True,

    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
sample_text = """The thing chess.com and lichess use isn't really a "generative model" in the deep learning sense, like an image or language model. It's closer to a search-based pipeline built around a chess engine (Stockfish) running over a huge database of real games. Lichess is fully open about this and even open-sources their generator, so it's a good reference point.

Here's the actual pipeline:

**Source positions from real games, not random generation.** You need a large database of played games (lichess publishes their entire game database for free, hundreds of millions of games with engine evaluations attached). Random or engine-vs-engine games work too, but puzzles from real games feel more natural because they come from genuine human mistakes.

**Run engine evaluation at every move of every game.** You walk through each game ply by ply with Stockfish at a reasonable depth, recording the centipawn evaluation after each move.

**Detect the blunder.** A puzzle starts where the evaluation swings sharply in one move, meaning one side just made a mistake that the other side can punish. A swing from roughly equal to a clear winning advantage (or from a small edge to a forced mate) is the signal you're looking for.

**Verify the punishing line is forced.** This is the part that actually makes something a good puzzle rather than just "a position where someone blundered." You run a multi-PV search (asking the engine for its top several candidate moves, not just the best one) at each step of the punishing sequence. If there's a large gap between the best move and the second-best move at every step, the line is forced and unambiguous, which is what makes a puzzle solvable with a single correct answer rather than several reasonable ones. If at any point two different moves are nearly as good as each other, the puzzle gets rejected or truncated, since a puzzle with multiple acceptable solutions is a bad puzzle.

**Tag themes.** Things like fork, pin, skewer, discovered attack, sacrifice, or mateInN are mostly detected with rule-based heuristics on the position and move sequence (checking piece movement patterns, whether material is given up before being recouped, whether the king is in check, etc.) rather than a trained classifier, though some projects do train small classifiers for this.

**Assign difficulty after the fact, not at generation time.** This is the part people often assume is baked into generation, but it isn't. Lichess treats every puzzle attempt like a mini chess game and runs a Glicko-2 rating system on it: the puzzle has a rating, you have a rating, and solving or failing it adjusts both, the same way winning or losing a game adjusts player ratings. So difficulty emerges from aggregate solver behavior over time, not from a formula applied to the position itself.

If you want to actually build something like this yourself, the tools are: `python-chess` for board representation and move generation, the Stockfish engine binary, and either the public lichess game database or your own collection of PGN files. Lichess's actual generator code is open source under `lichess-org/lila` (specifically the puzzler module), so you can read real production code rather than guessing at the algorithm.

A minimal version of the blunder-detection and forced-line-verification logic looks roughly like this in Python:The two thresholds (`EVAL_SWING_THRESHOLD` for detecting a blunder and `FORCED_GAP_THRESHOLD` for verifying the line is forced) are the knobs that determine puzzle quality versus quantity. Tighter thresholds give you fewer but cleaner puzzles; looser ones give you volume but more puzzles with ambiguous or multiple solutions, which is exactly the complaint people have when a generated puzzle "has two valid answers."

A few extensions worth knowing about if you go further with this. Theme tagging can be layered on by inspecting the solution line: a piece landing on a square attacked by two enemy pieces simultaneously is a fork, a piece that can't move without exposing the king to check is a pin, material given up in the first move and recouped later with extra is a sacrifice, and so on. For variety beyond captured games, you can also generate positions from engine self-play instead of human games, or use retrograde analysis to construct exact mate-in-N studies from scratch, though those tend to feel more artificial than puzzles pulled from real blunders. And if you want the difficulty rating to behave like chess.com's, you'd need actual solvers attempting the puzzles so the Glicko system has data to converge on, since there's no way to derive a meaningful human-difficulty number from engine analysis alone."""


inputs = tokenizer("summarize: "+ sample_text,return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(inputs["input_ids"], max_length=150, num_beams=4, early_stopping = True)

print("Generated Summary: ", tokenizer.decode(outputs[0], skip_special_token=True))


In [ ]:
trainer.save_model("./t5-small-cnn-dailymail")
tokenizer.save_pretrained("./t5-small-cnn-dailymail")

In [ ]:
from huggingface_hub import login
login(token="<token>")

In [ ]:
from huggingface_hub import HfApi, create_repo

repo_id = "Achiket/t5-small-cnn-dailymail-summarization"

create_repo(repo_id, exist_ok=True)

api = HfApi()

api.upload_folder(
    folder_path="./t5-small-cnn-dailymail",
    repo_id=repo_id,
)